# 001 RAG Architecture Overview

这是 RAG 知识库学习线的第一份 Notebook。

配套设计文档：

- `rag_knowledge_base_design.md`

本课不连接 MinIO、ES、Neo4j，也不调用大模型。第一课只做一件事：建立完整知识库链路的心智模型。

学习目标：

1. 区分知识转化和知识召回。
2. 理解普通 RAG、Hybrid RAG、GraphRAG 的区别。
3. 理解 MinIO、ES、Neo4j 在系统中的职责边界。
4. 用小型 Python 数据结构模拟 PDF、chunk、三元组和证据包。
5. 明确后续课程每一步要产出什么。

## 0. 本课程样例 PDF

后续 RAG 课程统一使用这份 PDF 作为样例文档：

```text
raw/北京市密云水库防御洪水方案.pdf
```

第一课只检查文件路径，不解析 PDF。第二课会开始接入 MinIO 和 PyMuPDF。


In [4]:
from pathlib import Path

SAMPLE_PDF = Path("raw/北京市密云水库防御洪水方案.pdf")

print("sample pdf:", SAMPLE_PDF)
print("exists:", SAMPLE_PDF.exists())
if SAMPLE_PDF.exists():
    print("size MB:", round(SAMPLE_PDF.stat().st_size / 1024 / 1024, 2))


sample pdf: raw/北京市密云水库防御洪水方案.pdf
exists: True
size MB: 5.24


## 1. 先建立整体心智模型

这条学习线不是只做一个向量检索 demo，而是做一套从 PDF 到问答的知识库流水线。

可以把它分成两大阶段：

```text
知识转化：把 PDF 变成可检索、可关联、可引用的知识资产。
知识召回：根据用户问题，从文本索引和图谱中找证据，再生成回答。
```

完整流程：

```text
PDF
-> MinIO 保存原文
-> PDF 解析
-> 文本清洗
-> Chunk 切片
-> 可选摘要
-> Embedding 入 ES
-> 三元组抽取
-> Neo4j 入库
-> BM25 + 向量 + 图检索
-> 证据融合
-> 问答回答
```

如果用 Java 后端类比：

```text
MinIO  像文件服务 / 对象存储
ES     像搜索引擎 + 向量索引
Neo4j  像关系非常灵活的图数据库
模型网关 像一个远程智能服务
Notebook 像每一步的可运行单元测试和教学脚本
```

In [5]:
PIPELINE = [
    {'stage': 'ingest', 'name': '上传 PDF 到 MinIO', 'output': 'raw/{doc_id}.pdf'},
    {'stage': 'parse', 'name': '解析 PDF 文本', 'output': 'pages.json'},
    {'stage': 'chunk', 'name': '清洗并切片', 'output': 'chunks.json'},
    {'stage': 'summary', 'name': '生成可选摘要', 'output': 'summary 字段'},
    {'stage': 'embedding', 'name': '生成向量并写入 ES', 'output': 'rag_chunks index'},
    {'stage': 'triple', 'name': '抽取实体和三元组', 'output': 'triples.json'},
    {'stage': 'graph', 'name': '写入 Neo4j 图谱', 'output': 'Document / Chunk / Entity / Relation'},
    {'stage': 'retrieval', 'name': '多路召回', 'output': 'text_evidence + graph_evidence'},
    {'stage': 'answer', 'name': '基于证据回答', 'output': '带引用的答案'},
]

for index, step in enumerate(PIPELINE, start=1):
    print(f"{index:02d}. [{step['stage']}] {step['name']} -> {step['output']}")

01. [ingest] 上传 PDF 到 MinIO -> raw/{doc_id}.pdf
02. [parse] 解析 PDF 文本 -> pages.json
03. [chunk] 清洗并切片 -> chunks.json
04. [summary] 生成可选摘要 -> summary 字段
05. [embedding] 生成向量并写入 ES -> rag_chunks index
06. [triple] 抽取实体和三元组 -> triples.json
07. [graph] 写入 Neo4j 图谱 -> Document / Chunk / Entity / Relation
08. [retrieval] 多路召回 -> text_evidence + graph_evidence
09. [answer] 基于证据回答 -> 带引用的答案


## 2. 普通 RAG、Hybrid RAG、GraphRAG 的区别

先把三个概念分清楚。

| 类型 | 主要解决什么 | 典型能力 | 局限 |
|---|---|---|---|
| 普通 RAG | 找语义相关片段 | 向量检索 | 精确词、编号、实体名可能不稳 |
| Hybrid RAG | 同时照顾关键词和语义 | BM25 + 向量 + RRF | 仍然主要是在找片段 |
| GraphRAG | 处理实体关系和跨文档关联 | 图谱查询 + 路径分析 | 需要前置抽取和图谱质量控制 |

本课程选择：

```text
Hybrid RAG + GraphRAG
```

也就是：

```text
ES 负责找相关文本片段。
Neo4j 负责找实体和关系。
模型负责抽取、压缩、综合，但不能替代证据。
```

In [6]:
rag_modes = {
    '普通 RAG': ['vector_search'],
    'Hybrid RAG': ['bm25_search', 'vector_search', 'rrf_fusion'],
    'GraphRAG': ['entity_extract', 'graph_query', 'path_reasoning'],
    '本课程组合': ['bm25_search', 'vector_search', 'rrf_fusion', 'graph_query', 'evidence_answer'],
}

for mode, capabilities in rag_modes.items():
    print(mode)
    for item in capabilities:
        print('  -', item)

普通 RAG
  - vector_search
Hybrid RAG
  - bm25_search
  - vector_search
  - rrf_fusion
GraphRAG
  - entity_extract
  - graph_query
  - path_reasoning
本课程组合
  - bm25_search
  - vector_search
  - rrf_fusion
  - graph_query
  - evidence_answer


## 3. 三个中间件的职责边界

这一点很重要。不要让一个组件承担所有职责。

| 组件 | 保存什么 | 不负责什么 |
|---|---|---|
| MinIO | PDF 原文、中间 JSON 产物 | 不负责检索、不负责推理 |
| ES | chunk 文本、summary、embedding、元数据 | 不负责复杂关系推理 |
| Neo4j | 实体、关系、证据、路径 | 不负责全文检索主链路 |

推荐边界：

```text
MinIO 是仓库。
ES 是搜索引擎。
Neo4j 是关系网络。
大模型是抽取和综合工具。
```

如果把所有逻辑都塞进向量检索，跨文档关系会很弱；如果把全文检索都塞进图数据库，又会失去搜索引擎的优势。

In [7]:
components = {
    'MinIO': {
        'stores': ['raw PDF', 'pages.json', 'chunks.json', 'triples.json'],
        'role': '对象存储，保证原始材料和中间产物可追溯',
    },
    'Elasticsearch': {
        'stores': ['chunk text', 'summary', 'embedding vector', 'metadata'],
        'role': '关键词检索和向量检索',
    },
    'Neo4j': {
        'stores': ['Document node', 'Chunk node', 'Entity node', 'RELATION edge'],
        'role': '实体关系、证据链、跨文档路径',
    },
}

for name, info in components.items():
    print(f"[{name}] {info['role']}")
    print('  stores:', ', '.join(info['stores']))

[MinIO] 对象存储，保证原始材料和中间产物可追溯
  stores: raw PDF, pages.json, chunks.json, triples.json
[Elasticsearch] 关键词检索和向量检索
  stores: chunk text, summary, embedding vector, metadata
[Neo4j] 实体关系、证据链、跨文档路径
  stores: Document node, Chunk node, Entity node, RELATION edge


## 4. 先用小数据模拟知识转化

后续课程会从 PDF 解析开始。这里先不用真实 PDF，用一个短文本模拟 PDF 中的一页。

这一页会被转换成：

```text
Page -> Chunk -> Triple
```

In [8]:
from pprint import pprint

page = {
    'doc_id': 'doc_001',
    'file_name': '密云水库防汛资料.pdf',
    'page_no': 1,
    'text': '密云水库管理处发布泄洪通知，通知要求下游乡镇做好防汛准备。密云区防汛办负责协调相关单位。',
}

chunk = {
    'doc_id': page['doc_id'],
    'chunk_id': 'doc_001_chunk_0001',
    'page_start': 1,
    'page_end': 1,
    'text': page['text'],
    'summary': '本段说明密云水库泄洪通知的发布单位、防汛要求和协调单位。',
    'metadata': {
        'file_name': page['file_name'],
        'source_object': 'raw/doc_001.pdf',
    },
}

triple = {
    'subject': '密云水库管理处',
    'predicate': '发布',
    'object': '泄洪通知',
    'evidence': '密云水库管理处发布泄洪通知',
    'confidence': 0.95,
    'doc_id': chunk['doc_id'],
    'chunk_id': chunk['chunk_id'],
}

pprint(page)
print('-' * 80)
pprint(chunk)
print('-' * 80)
pprint(triple)

{'doc_id': 'doc_001',
 'file_name': '密云水库防汛资料.pdf',
 'page_no': 1,
 'text': '密云水库管理处发布泄洪通知，通知要求下游乡镇做好防汛准备。密云区防汛办负责协调相关单位。'}
--------------------------------------------------------------------------------
{'chunk_id': 'doc_001_chunk_0001',
 'doc_id': 'doc_001',
 'metadata': {'file_name': '密云水库防汛资料.pdf', 'source_object': 'raw/doc_001.pdf'},
 'page_end': 1,
 'page_start': 1,
 'summary': '本段说明密云水库泄洪通知的发布单位、防汛要求和协调单位。',
 'text': '密云水库管理处发布泄洪通知，通知要求下游乡镇做好防汛准备。密云区防汛办负责协调相关单位。'}
--------------------------------------------------------------------------------
{'chunk_id': 'doc_001_chunk_0001',
 'confidence': 0.95,
 'doc_id': 'doc_001',
 'evidence': '密云水库管理处发布泄洪通知',
 'object': '泄洪通知',
 'predicate': '发布',
 'subject': '密云水库管理处'}


## 5. 为什么不能只做向量检索

如果用户问：

```text
泄洪通知是谁发布的？通知要求谁做什么？
```

向量检索可以找到相关 chunk，但它只返回文本片段。

图谱可以直接表达关系：

```text
密云水库管理处 --发布--> 泄洪通知
泄洪通知 --要求--> 下游乡镇做好防汛准备
密云区防汛办 --负责协调--> 相关单位
```

所以文本检索和图检索不是替代关系，而是互补关系。

In [9]:
graph_edges = [
    ('密云水库管理处', '发布', '泄洪通知'),
    ('泄洪通知', '要求', '下游乡镇做好防汛准备'),
    ('密云区防汛办', '负责协调', '相关单位'),
]

question = '泄洪通知是谁发布的？通知要求谁做什么？'

print('问题:', question)
print('图谱中可直接命中的关系:')
for subject, predicate, obj in graph_edges:
    if '泄洪通知' in (subject, obj):
        print(f'  {subject} --{predicate}--> {obj}')

问题: 泄洪通知是谁发布的？通知要求谁做什么？
图谱中可直接命中的关系:
  密云水库管理处 --发布--> 泄洪通知
  泄洪通知 --要求--> 下游乡镇做好防汛准备


## 6. 问答阶段使用证据包

后续不要直接把所有召回结果丢给模型。更好的做法是先组织证据包。

证据包包含两部分：

```text
text_evidence: ES 召回出来的文本片段
图谱 evidence: Neo4j 查出来的实体关系
```

模型回答时只能基于证据包回答。

In [10]:
evidence_package = {
    'question': '泄洪通知是谁发布的？通知要求谁做什么？',
    'text_evidence': [
        {
            'chunk_id': chunk['chunk_id'],
            'page_start': chunk['page_start'],
            'page_end': chunk['page_end'],
            'text': chunk['text'],
        }
    ],
    'graph_evidence': [
        {
            'subject': triple['subject'],
            'predicate': triple['predicate'],
            'object': triple['object'],
            'evidence': triple['evidence'],
            'chunk_id': triple['chunk_id'],
        },
        {
            'subject': '泄洪通知',
            'predicate': '要求',
            'object': '下游乡镇做好防汛准备',
            'evidence': '通知要求下游乡镇做好防汛准备',
            'chunk_id': chunk['chunk_id'],
        },
    ],
}

pprint(evidence_package)

{'graph_evidence': [{'chunk_id': 'doc_001_chunk_0001',
                     'evidence': '密云水库管理处发布泄洪通知',
                     'object': '泄洪通知',
                     'predicate': '发布',
                     'subject': '密云水库管理处'},
                    {'chunk_id': 'doc_001_chunk_0001',
                     'evidence': '通知要求下游乡镇做好防汛准备',
                     'object': '下游乡镇做好防汛准备',
                     'predicate': '要求',
                     'subject': '泄洪通知'}],
 'question': '泄洪通知是谁发布的？通知要求谁做什么？',
 'text_evidence': [{'chunk_id': 'doc_001_chunk_0001',
                    'page_end': 1,
                    'page_start': 1,
                    'text': '密云水库管理处发布泄洪通知，通知要求下游乡镇做好防汛准备。密云区防汛办负责协调相关单位。'}]}


## 7. 从证据包生成回答

第一课不调用大模型，我们先手写一个回答模板，观察“带证据回答”的结构。

后续第 8 课会把这一步替换成模型调用。

In [11]:
def answer_from_evidence(package: dict) -> str:
    relations = package["graph_evidence"]
    lines = ["根据当前证据："]
    for relation in relations:
        lines.append(
            f"- {relation['subject']} {relation['predicate']} {relation['object']}。"
            f"证据：{relation['evidence']}。来源：{relation['chunk_id']}"
        )
    return "\n".join(lines)

print(answer_from_evidence(evidence_package))


根据当前证据：
- 密云水库管理处 发布 泄洪通知。证据：密云水库管理处发布泄洪通知。来源：doc_001_chunk_0001
- 泄洪通知 要求 下游乡镇做好防汛准备。证据：通知要求下游乡镇做好防汛准备。来源：doc_001_chunk_0001


## 8. 本课程的后续路线

接下来每一课只推进一个能力：

1. `002-pdf-to-minio-and-parse.ipynb`：PDF 上传 MinIO，并解析 pages。
2. `003-clean-and-chunk-pdf-text.ipynb`：清洗文本并切片。
3. `004-summary-and-embedding-to-es.ipynb`：摘要、embedding、写入 ES。
4. `005-triple-extraction-with-qwen.ipynb`：使用 `qwen2.5-0.5b-instruct` 抽取三元组。
5. `006-write-graph-to-neo4j.ipynb`：把实体关系写入 Neo4j。
6. `007-hybrid-retrieval-plus-graph.ipynb`：BM25 + 向量 + 图检索。
7. `008-qa-with-evidence-and-validation.ipynb`：基于证据回答并做验证。

第一课的核心结论：

```text
RAG 不是只有向量检索。
知识库工程要先把知识转化成可检索、可关联、可验证的结构。
ES 解决找文本，Neo4j 解决找关系，模型负责抽取和综合。
```

## 9. 练习

请你用自己的话回答三个问题：

1. 为什么 MinIO 不应该负责检索？
2. 为什么 ES 不适合单独解决跨文档关系问题？
3. 为什么三元组关系必须保存 `evidence` 和 `chunk_id`？

下一课我们开始接入真实 MinIO 和 PDF 解析。